In [20]:
import copernicusmarine 
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json
import sys
import importlib
from datetime import datetime
from pathlib import Path
pytools_dir = Path.cwd() / "../pytools"
if str(pytools_dir) not in sys.path:
    sys.path.append(str(pytools_dir))
import icecalc
importlib.reload(icecalc)

##################################################################
cred_file = "~/.copernicusmarine/.copernicusmarine-credentials"
obs_dict = {
            'ice_extent_nh': {
                "dataset_id": "arctic_omi_si_extent_obs",
                "dataset_var": ["siextent"]},
            'ice_extent_sh': {
                "dataset_id": "antarctic_omi_si_extent_obs",
                "dataset_var": ["siextent"]},
            'ice_conc_nh': {
                "dataset_id": "osisaf_obs-si_glo_phy-sic-north_nrt_amsr2_l4_P1D-m",
                "dataset_var": ["ice_conc"]},
            'ice_conc_sh': {
                "dataset_id": "osisaf_obs-si_glo_phy-sic-south_nrt_amsr2_l4_P1D-m",
                "dataset_var": ["ice_conc"]},
            'ice_conc_nrt': {
                "dataset_id": "METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2",
                "dataset_var": ["sea_ice_fraction", "mask"]},
            'SST': {
                "dataset_id": "METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2",
                "dataset_var": ["analysed_sst", "sea_ice_fraction", "mask"]}
           }

##################################################################
def parse_copernicus_metadata(i):
    """
    Parses a CopernicusMarineCatalogue object into readable text and DataFrames.
    """
    print("########################################################################")
    print("dataset_id:", i)
    ds = copernicusmarine.open_dataset(dataset_id=obs_dict[i]['dataset_id'], credentials_file=cred_file)
    print("Min Time:", ds['time'].min().values)
    print("Max Time:", ds['time'].max().values)
    vars_list = []
    for var in ds.variables:
        print("Variable:", var)
        print("     dims:", ds[var].dims)
        for key, value in ds[var].attrs.items():
            print(f"     {key}: {value}")
    print("\n\n\n")
    return ds

##################################################################
def calc_cell_area(ds):
    R = 6371000.0  
    lats = ds.latitude.values
    lons = ds.longitude.values
    dlat_rad = np.radians(np.abs(np.mean(np.diff(lats))))
    dlon_rad = np.radians(np.abs(np.mean(np.diff(lons))))
    # Convert latitudes to radians and clamp bounds to [-90, 90]
    lats_clamped = np.clip(lats, -90.0, 90.0)
    lat_rad = np.radians(lats_clamped)
    # Calculate 1D area (strictly non-negative)
    cell_area_1d = (R**2) * dlat_rad * dlon_rad * np.cos(lat_rad)
    # Broadcast to 2D matching your dataset
    cell_area_2d = np.tile(cell_area_1d[:, np.newaxis], (1, len(lons)))
    cell_area = xr.DataArray(data=cell_area_2d / 1e6,  # Convert m² to km²
        coords={"latitude": ds.latitude, "longitude": ds.longitude},
        dims=["latitude", "longitude"])
    return cell_area

##################################################################
def arctic_plot(da):
    fig = plt.figure(figsize=(10, 10))
    ax = plt.subplot(1, 1, 1, projection=ccrs.NorthPolarStereo())
    # Set boundary extent to cover the Arctic domain (e.g., down to 50°N)
    ax.set_extent([-180, 180, 55, 90], crs=ccrs.PlateCarree())
    # 4. Plot siconc using Cartopy's coordinate transform
    # Using 'Blues_r' or 'cividis' works well for sea ice concentration
    p = da.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(),
        x='longitude', y='latitude',cmap='Blues_r',
        #vmin=0, vmax=1,  # Adjust to 1.0 if siconc is fractional (0 to 1)
        cbar_kwargs={
            'label': da.attrs['standard_name'],
            'orientation': 'horizontal',
            'pad': 0.05,
            'shrink': 0.8,},)
    # 5. Add geographic features
    #ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=2)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5, zorder=3)
    ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    # 6. Title and formatting
    valid_time_str = str(da.time.values)[:16]
    plt.title(
        f'Arctic Sea Ice Concentration (siconc)\nValid: {valid_time_str}',
        fontsize=14,
        pad=15,
    )
    plt.tight_layout()

##################################################################
def main():
    xr.set_options(keep_attrs=True)
    #ids = obs_dict.keys()
    ids = ['ice_extent_nh']
    for i in ids:
        ds = parse_copernicus_metadata(i)
    exit(1)
    v = 'ice_conc'
    #v = 'status_flag'
    #v = 'raw_ice_conc_values'
    da = ds[v].isel(time = 0)
    if v in ['ice_conc']:
        da = da / 100.0
        da.loc[da.latitude > 88] = 1
    print("########################################################################")
    print('ice extent NH')
    icecalc.extent.ds = ds.isel(time = 0)
    
    icecalc.extent.grid_size = calc_cell_area(ds)
    extent = icecalc.extent.calc()
    print(extent)
    print(extent.values)
    print("########################################################################")
    print('Arctic Plot')
    arctic_plot(da)
    print('NPB: done')
    ####

if __name__ == "__main__":
    main()

########################################################################
dataset_id: ice_extent_nh


INFO - 2026-10-20T21:04:13Z - Selected dataset version: "202511"
INFO - 2026-10-20T21:04:13Z - Selected dataset part: "default"
WARNING - 2026-10-20T21:04:13Z - You are using the dataset arctic_omi_si_extent_obs, version '202511', part 'default'. This exact version and part of the dataset will be retired on the 2026-11-24T13:00:00.000Z. For more information you can check: https://marine.copernicus.eu/user-corner/product-roadmap/transition-information


Min Time: 1979-01-01T00:00:00.000000000
Max Time: 2024-12-01T00:00:00.000000000
Variable: monthlyclimsiextent_mean
     dims: ('clim_months',)
     cell_methods: time: mean within years time: mean over years
     long_name: Monthly climatology of the Sea Ice Extent
     standard_name: sea_ice_extent
     units: km^2
Variable: monthlyclimsiextent_std
     dims: ('clim_months',)
     cell_methods: time: standard_deviation within years time: standard_deviation over years
     long_name: Standard deviation of the monthly climatology of Sea Ice Extent
     units: km^2
Variable: siextent
     dims: ('time',)
     comment: Lake ice is not included
     long_name: Monthly averaged Sea Ice Extent
     standard_name: sea_ice_extent
     threshold: sea_ice_area_fraction_threshold
     units: km^2
Variable: clim_months
     dims: ('clim_months',)
     climatology: clim_months_bounds
     comment: The year is that of the middle of the first interval in clim_months_bounds
     long_name: monthly cli

KeyError: "No variable named 'ice_conc'. Variables on the dataset include ['monthlyclimsiextent_mean', 'monthlyclimsiextent_std', 'siextent', 'clim_months', 'time']"